# AEGIS Cloud Video Studio (100% Powered by AEGIS NEXUS + Free Kaggle GPU)
**سحب الفيديوهات (YouTube/روابط) + تفريغ وترجمة مباشرة من AEGIS + صوتيات متعددة الشخصيات + رندر سحابي بدون تسخين الحاسوب**
1. Turn ON **GPU T4 x2** & **Internet** in Kaggle right sidebar (or even CPU works cleanly since AEGIS handles transcription, translation & TTS!).
2. Add Secret `AISTUDIO_KEYS_JSON` in **Add-ons → Secrets** (or set `AEGIS_BASE_URL` if using Cloudflare Quick Tunnel).
3. Run Cell 1 & Cell 2, then run **Mode A** (Pull YouTube/Video → AEGIS Extract & Translate → Cinema `.ass` + Multi-Voice Neural Dub + MP3) or **Mode B** (Fun Storytelling 30fps Motion Video + SFX/Soundbed + Podcast MP3).

In [ ]:
%%bash
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'CPU mode'
apt-get update -qq && apt-get install -y -qq ffmpeg fonts-noto-core fonts-noto-extra fontconfig > /dev/null
fc-cache -f
pip install -q playwright imageio-ffmpeg yt-dlp
playwright install chromium
playwright install-deps chromium > /dev/null 2>&1 || true
ffmpeg -hide_banner -encoders 2>/dev/null | grep -E 'h264_nvenc|libx264' | head -n 5

In [ ]:
import os, sys
from pathlib import Path

STUDIO_DIR = Path('/kaggle/working/aegis_cloud_video_studio')
if STUDIO_DIR.exists() and str(STUDIO_DIR) not in sys.path:
    sys.path.insert(0, str(STUDIO_DIR))

try:
    from kaggle_secrets import UserSecretsClient
    os.environ['AISTUDIO_KEYS_JSON'] = UserSecretsClient().get_secret('AISTUDIO_KEYS_JSON')
    print('Loaded AISTUDIO_KEYS_JSON securely from Kaggle Secrets.')
except Exception as exc:
    print('Secret check:', exc, '-> set AEGIS_BASE_URL or AISTUDIO_KEYS_JSON manually if needed.')

from aegis_studio import AegisStudioClient
client = AegisStudioClient()
print(f'AEGIS Client ready | mode={client.mode} | keys_loaded={len(client.keys)}')

In [ ]:
# === MODE A: Pull Video (YouTube URL / Kaggle file) -> AEGIS Extract & Translate + Multi-Voice Dub ===
from pathlib import Path
from aegis_studio.subtitle_dub_pipeline import run_subtitle_dub_pipeline

SOURCE_VIDEO = 'https://www.youtube.com/watch?v=YOUR_VIDEO_ID'  # Paste YouTube/URL or /kaggle/input/... path
if 'YOUR_VIDEO_ID' not in SOURCE_VIDEO:
    result_a = run_subtitle_dub_pipeline(
        video_input=SOURCE_VIDEO,
        out_dir=Path('/kaggle/working/output_subtitled'),
        target_lang='العربية',
        preset='cinema',
        asr_engine='aegis',     # Pure AEGIS NEXUS audio pull + translation (zero Whisper model download needed!)
        dual_language=True,
        enable_dubbing=True,    # Multi-speaker Arabic/French neural dubbing via Gemini 3.8 TTS + audio ducking!
        voice_name='Charon',
        multi_voice=True,
        client=client,
    )
    print(result_a)
else:
    print('Replace SOURCE_VIDEO with a YouTube URL or local MP4 path to run Mode A.')

In [ ]:
# === MODE B: AEGIS Fun Storytelling Video (Multi-Layout 30fps Motion + 2-Voice Cast + Soundbed + MP3) ===
from pathlib import Path
from aegis_studio.motion_video_pipeline import run_motion_video_pipeline

result_b = run_motion_video_pipeline(
    topic='قصة مثيرة عن كيف غيّر الذكاء الاصطناعي صناعة السينما والترجمة في العالم العربي',
    out_dir=Path('/kaggle/working/output_motion'),
    language='العربية',
    scene_count=5,
    style_mode='storytelling',  # Alternating hosts (Charon + Aoede) + hook_impact / versus / timeline layouts
    width=1920,
    height=1080,
    fps=30,
    voice_default='Charon',
    with_captions=True,
    with_sfx=True,
    client=client,
)
print(result_b)